# GPT G5 — cold-start compiler workers

Experimental branch only. Fresh caches for every official harness run. Compare the original frozen-v4 thread against 1 and 2 compiler subprocesses in a balanced order, plus a separate diagnostic of warmup completion versus first step. The public v4 release is unchanged.

In [ ]:
import os,sys,subprocess,pathlib,json,hashlib,shutil,statistics,importlib.metadata as md
ROOT=pathlib.Path("/kaggle/working")
REPO=ROOT/"repo";ORIG=ROOT/"v4"
COMMIT="dfe0ad612265c02d7ded3bd8a99037f943fa87a3"
FROZEN="5c1bbe560c08b0323eefeb843a0baf00e400e774"
V51="d9ab004"
packages=["soup-cli[train]==0.77.0","transformers==5.16.1","peft==0.20.0","trl==0.29.1","bitsandbytes==0.50.2","accelerate==1.14.0","pytest"]
subprocess.run([sys.executable,"-m","pip","install","-q",*packages],check=True)
assert md.version("torch").startswith("2.11.0") and md.version("triton")=="3.6.0"
for repo,sha in [(REPO,COMMIT),(ORIG,FROZEN)]:
    shutil.rmtree(repo,ignore_errors=True)
    subprocess.run(["git","clone","-q","https://github.com/tokha-t/decik.git",str(repo)],check=True)
    subprocess.run(["git","checkout","--detach","-q",sha],cwd=repo,check=True)
    subprocess.run([sys.executable,"task/prepare_data.py"],cwd=repo,check=True)
OUT=ROOT/"g5-evidence";OUT.mkdir(exist_ok=True)
def checked(cmd,cwd,tag,env=None):
    p=subprocess.run(cmd,cwd=cwd,env=env,capture_output=True,text=True)
    (OUT/(tag+".log")).write_text(p.stdout+"\n"+p.stderr)
    print(tag,p.stdout[-1800:],p.stderr[-800:],flush=True)
    p.check_returncode()
    return p
checked([sys.executable,"-m","unittest","discover","-s","dev","-p","test_warmup_process.py","-v"],REPO,"lifecycle")
checked([sys.executable,"dev/compile_sm75.py","."],REPO,"compile-sm75")
for mode in (1,2):
    env=dict(os.environ,SKC_WARMUP_PROCESSES=str(mode),TRITON_CACHE_DIR=str(ROOT/f"test-cache-{mode}"))
    checked([sys.executable,"-m","pytest","solution","-q"],REPO,f"gpu-{mode}",env)
    checked([sys.executable,"-m","pytest","tests","-q"],REPO,f"soup-{mode}",dict(env,SOUP_APPLY_SOLUTION="1"))
print("G5_PREFLIGHT_OK",flush=True)


In [ ]:
# Original baseline and candidate use the exact same workload, data, packages and GPU session.
summary=[]
def run(tag,repo,mode=None,side="solution"):
    env=dict(os.environ)
    for k in list(env):
        if k.startswith("SKC_"):env.pop(k)
    if mode is not None:env["SKC_WARMUP_PROCESSES"]=str(mode)
    run_dir=repo/("runs_"+tag)
    checked([sys.executable,"harness/run_task.py","--only",side,"--out",str(run_dir)],repo,tag,env)
    result=json.loads((run_dir/"result.json").read_text())
    data=result["runs"][side][0];assert data["exit_code"]==0
    target=OUT/tag;target.mkdir(exist_ok=True)
    for name in ("result.json","eval_"+side+".json"):
        shutil.copy2(run_dir/name,target/name)
    for name in ("train.log","metrics.json"):
        shutil.copy2(run_dir/(side+"_1")/name,target/name)
    row={"tag":tag,"mode":mode,"source":FROZEN if repo==ORIG else COMMIT,"wall_s":data["wall_s"],"peak_mib":data["gpu_peak_mib"],"loop_s":data["metrics"]["train_loop_s"],"apply_s":data["metrics"]["apply_s"],"first_step_s":json.loads((target/"metrics.json").read_text())["step_s"][0],"eval_loss":result["summary"][side]["eval_loss"]}
    summary.append(row);(OUT/"summary.json").write_text(json.dumps(summary,indent=2))
    print("G5_RESULT",json.dumps(row),flush=True)
run("base",ORIG,side="base")
for tag,repo,mode in [("thread-a",ORIG,None),("proc1-a",REPO,1),("proc2-a",REPO,2),("proc2-b",REPO,2),("proc1-b",REPO,1),("thread-b",ORIG,None)]:
    run(tag,repo,mode)
print("G5_AB_COMPLETE",json.dumps(summary),flush=True)


In [ ]:
# Separate diagnostic timestamps; these runs are not part of A/B wall-time scores.
sys.path.insert(0,str(REPO/"harness"))
from run_task import make_config,warm_up
from common import OFFLINE_ENV,run_logged
for mode in (0,1,2):
    rd=REPO/f"warmup_probe_{mode}";rd.mkdir(exist_ok=True)
    warm_up(rd)
    config=make_config(rd,REPO/"data/train.jsonl",1)
    timeline=OUT/f"timeline-{mode}.jsonl";timeline.unlink(missing_ok=True)
    env=dict(OFFLINE_ENV,SKC_WARMUP_PROCESSES=str(mode),SKC_WARMUP_TRACE=str(timeline),SOUP_DB_PATH=str(rd/"db.sqlite"),TRITON_CACHE_DIR=str(rd/"triton_cache"),TORCHINDUCTOR_CACHE_DIR=str(rd/"inductor_cache"))
    code,wall,peak,start=run_logged([sys.executable,str(REPO/"dev/warmup_probe.py"),"--config",str(config),"--metrics",str(rd/"metrics.json"),"--solution"],rd/"train.log",900,env)
    print("G5_PROBE",mode,code,wall,peak,flush=True)
    assert code==0,(rd/"train.log").read_text()[-5000:]
    events=[json.loads(line) for line in timeline.read_text().splitlines()]
    ready=[x for x in events if x["event"]=="ready"][-1]
    first=[x for x in events if x["event"]=="first_step_begin"][0]
    assert not [x for x in events if x["event"] in ("compile_error","process_error","worker_failed")],events
    compiled={x["kernel"] for x in events if x["event"]=="kernel_end" and x["pid"]==ready["pid"]}
    assert len(compiled)==11,compiled
    print("G5_TIMELINE",json.dumps({"mode":mode,"ready_before_first_step_s":(first["time_ns"]-ready["time_ns"])/1e9,"kernel_count":len(compiled),"metrics":json.loads((rd/"metrics.json").read_text())["step_s"][:3]}),flush=True)
    shutil.copy2(rd/"metrics.json",OUT/f"probe-{mode}-metrics.json")
    shutil.copy2(rd/"train.log",OUT/f"probe-{mode}-train.log")
# Verify the same warmup kernel coverage against v5.1 without editing its branch.
compat=ROOT/"v51"
shutil.copytree(ORIG,compat,ignore=shutil.ignore_patterns("runs_*","data"),dirs_exist_ok=True)
subprocess.run(["git","checkout","--detach","-q",V51],cwd=compat,check=True)
shutil.copy2(REPO/"solution/warmup.py",compat/"solution/warmup.py")
checked([sys.executable,"-m","pytest","solution","-q"],compat,"v51-overlay-gpu",dict(os.environ,SKC_WARMUP_PROCESSES="2"))
manifest={"candidate_commit":COMMIT,"frozen_v4":FROZEN,"v51_compatibility_overlay":V51,"summary":summary,"client_date":"2026-10-31","timezone":"Asia/Almaty","versions":{x:md.version(x) for x in ("torch","triton","soup-cli","transformers","peft","trl","bitsandbytes","accelerate")},"sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob("*") if p.is_file()}}
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
shutil.make_archive("/kaggle/working/g5-evidence","zip",OUT)
print("G5_COMPLETE",json.dumps(manifest),flush=True)
